# Comics vs. COCO: Do the Same Heads Vir on Both?

This notebook runs vir-head **discovery** on a single Qwen-VL checkpoint against two
different datasets:

- **Comics** — six-panel comic strips, scored by `vis_head.vir.aggregate_region_attention`
  (mean raw attention the final prompt token places on the *queried* panel's image
  tokens; see `01_discover_vis_heads.py`).
- **COCO** — the COCO vir dataset from `build_coco_vis_head_dataset.ipynb`
  (`"Find the <category>."` + full image), scored by
  `vis_head.coco.vis_head_score_from_patch_mask` (attention weighted by per-patch object
  occupancy from the COCO segmentation).

It has four parts:

1. **Raw vis-head-score comparison** — top-k stats, overlap, Wilcoxon test, plots.
2. **Area-normalized ("fair") vir score** — comic panels cover ~1/N of the image while
   a COCO object is typically much smaller, so raw attention *mass* mechanically favors
   comics even for a head with identical targeting behavior. We divide each sample's
   raw score by that sample's target-region size (as a fraction of image tokens),
   turning "how much attention mass" into "how much attention *density relative to
   chance*" — an enrichment ratio that is comparable across region sizes and datasets.
3. **Causal-effect comparison** — for a shared set of top vis heads, we *ablate* them
   (force their attention away from the target) and measure how much the generated
   text actually changes, per dataset. This asks a different question than the
   attention-score comparison: not just "do these heads look at the target more than
   chance," but "does redirecting them away from the target actually change the
   model's behavior" — and on which dataset that causal effect is larger.
4. **Verdict.**

**Prerequisites**
- Comics under `VIR_COMICS_ROOT` (`comicN/p1..pN` folders).
- A COCO vir dataset built by `build_coco_vis_head_dataset.ipynb` /
  `06_build_coco_vis_head_dataset.py` under `data/coco_vis_head/`.
- A GPU with enough memory for one Qwen-VL checkpoint.
- Run from the repository root so `vis_head` imports resolve.


In [1]:
%matplotlib inline
import gc
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vis_head").exists():
    raise RuntimeError("Run this notebook from the repository root (vis-head/).")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from scipy import stats
from tqdm.auto import tqdm

from vis_head.common import (
    DEFAULT_COMICS_ROOT,
    DEFAULT_MODEL_ID,
    DEFAULT_N_PANELS,
    DEFAULT_SEED,
    dump_json,
    make_output_paths,
    seed_everything,
)
from vis_head.coco import (
    DEFAULT_COCO_OUTPUT_DIR,
    vis_head_score_from_patch_mask,
    load_coco_vis_head_dataset,
    target_weight_fraction,
)
from vis_head.data import build_strip, list_comic_dirs
from vis_head.vir import (
    aggregate_region_attention,
    collect_last_query_attentions,
    panel_query_prompt,
    panel_token_fractions,
    rank_heads_by_score,
    save_head_ranking,
)
from vis_head.judge import bootstrap_ci, semantic_similarity
from vis_head.modeling import (
    decode_generated_text,
    find_image_token_range,
    load_model_and_processor,
    model_dims,
    prepare_inputs,
    run_generation,
)
from vis_head.plots import save_figure_pdf
from vis_head.regions import assign_panels_to_tokens, bbox_to_token_positions, get_merged_grid_shape, region_positions_from_ids
from vis_head.steering import (
    group_heads_by_layer,
    make_static_attention_mask_hook,
    register_mask_hooks,
    remove_handles,
)


In [2]:
# ----------------------------- configuration -----------------------------
MODEL_ID = DEFAULT_MODEL_ID     # one checkpoint, evaluated on both datasets
DEVICE = "cuda:0"

COMICS_ROOT = Path(DEFAULT_COMICS_ROOT)
N_PANELS = DEFAULT_N_PANELS
N_COMIC_SAMPLES = 200            # comics to score

COCO_DATASET_DIR = DEFAULT_COCO_OUTPUT_DIR
N_COCO_SAMPLES = 200             # COCO samples to score

SEED = DEFAULT_SEED
OUTPUT_NAME_TEMPLATE = "vis_head_discovery_compare_datasets_{tag}"
EPS = 1e-8   # guards the /fraction division for near-empty regions

comic_dirs = list_comic_dirs(COMICS_ROOT, n_panels=N_PANELS)
if not comic_dirs:
    raise FileNotFoundError(f"No comicN/p1..p{N_PANELS} folders under {COMICS_ROOT}")
comic_dirs = comic_dirs[:N_COMIC_SAMPLES]

coco_dataset = load_coco_vis_head_dataset(COCO_DATASET_DIR)
if len(coco_dataset) == 0:
    raise FileNotFoundError(
        f"No samples in {COCO_DATASET_DIR} — run build_coco_vis_head_dataset.ipynb first."
    )
coco_indices = list(range(min(N_COCO_SAMPLES, len(coco_dataset))))

print(f"Model       : {MODEL_ID}")
print(f"Comics root : {COMICS_ROOT}  ({len(comic_dirs)} strips)")
print(f"COCO dataset: {COCO_DATASET_DIR}  ({len(coco_indices)} samples)")


Model       : Qwen/Qwen3-VL-8B-Instruct
Comics root : /mnt/abka03/Projects/vis-head/data/comics  (200 strips)
COCO dataset: /mnt/abka03/Projects/vis-head/data/coco_vis_head  (200 samples)


## Part 1 — Discovery

Both functions return **raw** scores (mean attention mass on the target, as before)
*and* **normalized** scores (that per-sample mass divided by the target region's token
fraction, then averaged) — see Part 2 for why.

In [3]:
def discover_vis_head_comics(model, processor, comic_dirs, n_panels: int):
    """Same procedure as 01_discover_vis_heads.py: the diagonal of the
    queried-panel x attended-panel matrix, averaged over comics. Also
    accumulates the area-normalized diagonal (raw / panel-token-fraction)."""
    n_layers, n_heads, spatial_merge = model_dims(model)
    vir_sum = np.zeros((n_panels, n_layers, n_heads, n_panels), dtype=np.float64)
    normalized_sum = np.zeros((n_panels, n_layers, n_heads), dtype=np.float64)
    valid_samples = 0

    for comic_dir in tqdm(comic_dirs, desc="Vir discovery [comics]"):
        strip = build_strip(comic_dir, n_panels=n_panels)
        region_ids = None
        fractions = None
        per_prompt = []
        ok = True
        for panel_index in range(1, n_panels + 1):
            prompt = panel_query_prompt(panel_index, n_panels=n_panels)
            try:
                inputs = prepare_inputs(processor, strip.strip, prompt, DEVICE)
                if region_ids is None:
                    region_ids, _, _ = assign_panels_to_tokens(
                        image_grid_thw=inputs["image_grid_thw"],
                        panel_widths=strip.panel_widths,
                        spatial_merge=spatial_merge,
                    )
                    fractions = panel_token_fractions(region_ids, n_panels)
                attn_at_query = collect_last_query_attentions(model, inputs)
                panel_attention = aggregate_region_attention(
                    attn_at_query=attn_at_query, inputs=inputs, processor=processor,
                    region_ids=region_ids, n_regions=n_panels,
                )
                per_prompt.append(panel_attention)
            except Exception as exc:
                print(f"Skipping {strip.name} panel {panel_index}: {exc}")
                ok = False
                break
        if not ok or len(per_prompt) != n_panels:
            continue
        for prompt_idx in range(n_panels):
            vir_sum[prompt_idx] += per_prompt[prompt_idx]
            normalized_sum[prompt_idx] += per_prompt[prompt_idx][:, :, prompt_idx] / max(fractions[prompt_idx], EPS)
        valid_samples += 1

    if valid_samples == 0:
        raise RuntimeError("No valid comic samples processed.")

    mean_panel_attention = vir_sum / float(valid_samples)
    vis_head_scores = np.zeros((n_layers, n_heads), dtype=np.float32)
    for layer_idx in range(n_layers):
        for head_idx in range(n_heads):
            diag = [mean_panel_attention[p, layer_idx, head_idx, p] for p in range(n_panels)]
            vis_head_scores[layer_idx, head_idx] = float(np.mean(diag))
    normalized_scores = (normalized_sum.mean(axis=0) / float(valid_samples)).astype(np.float32)

    print(f"valid samples: {valid_samples}/{len(comic_dirs)}")
    return vis_head_scores, normalized_scores, valid_samples


In [4]:
def discover_vis_head_coco(model, processor, coco_dataset, sample_indices):
    """One "Find the <category>." instruction per sample, scored against the
    COCO segmentation mapped onto the visual-patch grid (continuous per-patch
    occupancy weights, not a one-hot region id). Also accumulates the
    area-normalized score (raw / target-weight-fraction)."""
    n_layers, n_heads, spatial_merge = model_dims(model)
    score_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    normalized_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    valid_samples = 0

    for idx in tqdm(sample_indices, desc="Vir discovery [coco]"):
        meta, gt = coco_dataset[idx]
        try:
            image = Image.open(meta["image_path"]).convert("RGB")
            inputs = prepare_inputs(processor, image, meta["instruction"], DEVICE)
            img_start, img_end = find_image_token_range(inputs, processor)
            attn_at_query = collect_last_query_attentions(model, inputs)
            raw = vis_head_score_from_patch_mask(attn_at_query, img_start, img_end, gt["patch_mask_flat"])
            fraction = target_weight_fraction(gt["patch_mask_flat"])
        except Exception as exc:
            print(f"Skipping sample {idx} ({meta.get('image_path')}): {exc}")
            continue
        score_sum += raw
        normalized_sum += raw / max(fraction, EPS)
        valid_samples += 1

    if valid_samples == 0:
        raise RuntimeError("No valid COCO samples processed.")

    vis_head_scores = (score_sum / valid_samples).astype(np.float32)
    normalized_scores = (normalized_sum / valid_samples).astype(np.float32)
    print(f"valid samples: {valid_samples}/{len(sample_indices)}")
    return vis_head_scores, normalized_scores, valid_samples


### Run discovery on both datasets

One model load, one GPU pass over each dataset in turn.

In [5]:
model, processor = load_model_and_processor(model_id=MODEL_ID, device=DEVICE)
n_layers, n_heads, spatial_merge = model_dims(model)
print(f"{n_layers} layers x {n_heads} heads")

seed_everything(SEED)
comic_scores, comic_scores_norm, n_comic_valid = discover_vis_head_comics(model, processor, comic_dirs, n_panels=N_PANELS)

seed_everything(SEED)
coco_scores, coco_scores_norm, n_coco_valid = discover_vis_head_coco(model, processor, coco_dataset, coco_indices)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

36 layers x 32 heads


Vir discovery [comics]:   0%|          | 0/200 [00:00<?, ?it/s]

valid samples: 200/200


Vir discovery [coco]:   0%|          | 0/200 [00:00<?, ?it/s]

valid samples: 200/200


In [6]:
comic_ranked = rank_heads_by_score(comic_scores)
coco_ranked = rank_heads_by_score(coco_scores)
comic_ranked_norm = rank_heads_by_score(comic_scores_norm)
coco_ranked_norm = rank_heads_by_score(coco_scores_norm)

outputs = make_output_paths(OUTPUT_NAME_TEMPLATE.format(tag="comics"))
np.save(outputs.logs_dir / "vis_head_scores.npy", comic_scores)
np.save(outputs.logs_dir / "vis_head_scores_normalized.npy", comic_scores_norm)
save_head_ranking(outputs.logs_dir / "vis_head_ranking.json", comic_ranked)
save_head_ranking(outputs.logs_dir / "vis_head_ranking_normalized.json", comic_ranked_norm)
dump_json(outputs.logs_dir / "summary.json", {
    "model_id": MODEL_ID, "dataset_source": "comics",
    "n_valid_samples": n_comic_valid, "n_layers": n_layers, "n_heads": n_heads,
    "top_heads": comic_ranked[:20], "top_heads_normalized": comic_ranked_norm[:20],
})

outputs = make_output_paths(OUTPUT_NAME_TEMPLATE.format(tag="coco"))
np.save(outputs.logs_dir / "vis_head_scores.npy", coco_scores)
np.save(outputs.logs_dir / "vis_head_scores_normalized.npy", coco_scores_norm)
save_head_ranking(outputs.logs_dir / "vis_head_ranking.json", coco_ranked)
save_head_ranking(outputs.logs_dir / "vis_head_ranking_normalized.json", coco_ranked_norm)
dump_json(outputs.logs_dir / "summary.json", {
    "model_id": MODEL_ID, "dataset_source": "coco",
    "n_valid_samples": n_coco_valid, "n_layers": n_layers, "n_heads": n_heads,
    "top_heads": coco_ranked[:20], "top_heads_normalized": coco_ranked_norm[:20],
})
print("Saved rankings under logs/vis_head_discovery_compare_datasets_{comics,coco}/")


Saved rankings under logs/vis_head_discovery_compare_datasets_{comics,coco}/


## Part 2 — Raw vis-head-score comparison

`comic_scores` and `coco_scores` are both `(n_layers, n_heads)` vis-head-score matrices
from the *same* model, directly comparable head-for-head — but see the caveat below
before reading too much into the magnitude gap.

In [7]:
def top_k_stats(scores: np.ndarray, k: int) -> dict:
    flat = np.sort(scores.reshape(-1))[::-1]
    top = flat[:k]
    return {"mean": float(top.mean()), "max": float(flat[0])}

def top_k_head_set(ranked: list[dict], k: int) -> set:
    return {(row["layer"], row["head"]) for row in ranked[:k]}

rows = []
for k in (1, 10, 50, 100):
    c = top_k_stats(comic_scores, k)
    j = top_k_stats(coco_scores, k)
    rows.append((k, c["mean"], j["mean"]))

print(f"{'top-k':>6s}  {'comics mean':>12s}  {'coco mean':>10s}  {'winner':>10s}")
for k, c_mean, j_mean in rows:
    winner = "coco" if j_mean > c_mean else ("comics" if c_mean > j_mean else "tie")
    print(f"{k:6d}  {c_mean:12.4f}  {j_mean:10.4f}  {winner:>10s}")

print()
print(f"Overall mean vir score  | comics: {comic_scores.mean():.5f}  coco: {coco_scores.mean():.5f}")
print(f"Overall max  vir score  | comics: {comic_scores.max():.5f}  coco: {coco_scores.max():.5f}")

comic_frac_mean = float(np.mean([1.0 / N_PANELS] * 1))  # comic panels are ~1/N_PANELS of the image
coco_frac_mean = float(np.mean([target_weight_fraction(coco_dataset[i][1]["patch_mask_flat"]) for i in coco_indices]))
print(f"\nCAVEAT — mean target-region size as a fraction of image tokens: "
      f"comics ~{comic_frac_mean:.3f}  vs  coco ~{coco_frac_mean:.4f} "
      f"({comic_frac_mean / max(coco_frac_mean, EPS):.1f}x larger). "
      "Raw attention *mass* mechanically scales with region size, so this alone can "
      "make comics look like the 'higher-scoring' dataset even with identical "
      "per-head targeting behavior. Part 3 corrects for this.")


 top-k   comics mean   coco mean      winner
     1        0.7160      0.1365      comics
    10        0.6429      0.0912      comics
    50        0.4119      0.0469      comics
   100        0.3025      0.0343      comics

Overall mean vir score  | comics: 0.04759  coco: 0.00723
Overall max  vir score  | comics: 0.71596  coco: 0.13647

CAVEAT — mean target-region size as a fraction of image tokens: comics ~0.167  vs  coco ~0.1265 (1.3x larger). Raw attention *mass* mechanically scales with region size, so this alone can make comics look like the 'higher-scoring' dataset even with identical per-head targeting behavior. Part 3 corrects for this.


In [8]:
for k in (10, 50, 100):
    comic_top = top_k_head_set(comic_ranked, k)
    coco_top = top_k_head_set(coco_ranked, k)
    overlap = comic_top & coco_top
    print(f"top-{k:<4d} overlap: {len(overlap)}/{k}  ({100 * len(overlap) / k:.1f}%)")

spearman = stats.spearmanr(comic_scores.reshape(-1), coco_scores.reshape(-1))
print(f"\nSpearman rank correlation (comics vs coco, per head, RAW scores): "
      f"rho={spearman.correlation:.3f}, p={spearman.pvalue:.3e}")

wilcoxon = stats.wilcoxon(coco_scores.reshape(-1), comic_scores.reshape(-1))
print(f"Wilcoxon signed-rank test (coco vs comics, paired by head, RAW scores): "
      f"statistic={wilcoxon.statistic:.1f}, p={wilcoxon.pvalue:.3e}")


top-10   overlap: 3/10  (30.0%)
top-50   overlap: 19/50  (38.0%)
top-100  overlap: 36/100  (36.0%)

Spearman rank correlation (comics vs coco, per head, RAW scores): rho=0.748, p=5.342e-207
Wilcoxon signed-rank test (coco vs comics, paired by head, RAW scores): statistic=59454.0, p=1.039e-128


In [9]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
axes[0].hist(comic_scores.reshape(-1), bins=60, alpha=0.55, label="comics", color="tab:orange")
axes[0].hist(coco_scores.reshape(-1), bins=60, alpha=0.55, label="coco", color="tab:green")
axes[0].set_xlabel("Raw vir score"); axes[0].set_ylabel("Number of heads")
axes[0].set_title("Raw score distribution"); axes[0].legend()

axes[1].scatter(comic_scores.reshape(-1), coco_scores.reshape(-1), s=8, alpha=0.4, color="tab:green")
lim = max(float(comic_scores.max()), float(coco_scores.max())) * 1.05
axes[1].plot([0, lim], [0, lim], color="tab:red", linestyle="--", linewidth=1, label="y = x")
axes[1].set_xlim(0, lim); axes[1].set_ylim(0, lim)
axes[1].set_xlabel("Comics vir score"); axes[1].set_ylabel("COCO vir score")
axes[1].set_title("Per-head comparison (raw)"); axes[1].legend()

k = 100
comic_top, coco_top = top_k_head_set(comic_ranked, k), top_k_head_set(coco_ranked, k)
axes[2].bar(["comics only", "both", "coco only"],
           [len(comic_top - coco_top), len(comic_top & coco_top), len(coco_top - comic_top)],
           color=["tab:orange", "tab:gray", "tab:green"])
axes[2].set_title(f"Top-{k} head overlap (raw)"); axes[2].set_ylabel("Number of heads")

save_figure_pdf(fig, outputs.figures_dir / "comics_vs_coco_vis_head_scores_raw.pdf")
plt.show()


## Part 3 — Area-normalized ("fair") vir score

A head that does nothing but spread attention proportionally to region size already
scores well under raw attention mass — and a comic panel is typically ~1/6 of the
image while a COCO object is often a few percent, so raw scores mechanically favor
comics regardless of targeting behavior.

`normalized_score = raw_attention_mass_on_target / target_region_token_fraction`,
computed **per sample** (per panel query / per COCO object) before averaging. A score
of 1.0 means the head attends to the target exactly as much as its size would predict
by chance; a score >> 1 means the head concentrates attention on the target far beyond
what its size explains — the real signature of "vir" behavior, and now comparable
across region sizes and datasets.

In [10]:
rows = []
for k in (1, 10, 50, 100):
    c = top_k_stats(comic_scores_norm, k)
    j = top_k_stats(coco_scores_norm, k)
    rows.append((k, c["mean"], j["mean"]))

print("NORMALIZED (area-fair) scores")
print(f"{'top-k':>6s}  {'comics mean':>12s}  {'coco mean':>10s}  {'winner':>10s}")
for k, c_mean, j_mean in rows:
    winner = "coco" if j_mean > c_mean else ("comics" if c_mean > j_mean else "tie")
    print(f"{k:6d}  {c_mean:12.2f}  {j_mean:10.2f}  {winner:>10s}")

print()
print(f"Overall mean normalized score | comics: {comic_scores_norm.mean():.3f}  coco: {coco_scores_norm.mean():.3f}")
print(f"Overall max  normalized score | comics: {comic_scores_norm.max():.3f}  coco: {coco_scores_norm.max():.3f}")
print("(1.0 = attention proportional to region size / chance; higher = more concentrated than chance)")

for k in (10, 50, 100):
    overlap = top_k_head_set(comic_ranked_norm, k) & top_k_head_set(coco_ranked_norm, k)
    print(f"\ntop-{k:<4d} overlap (normalized): {len(overlap)}/{k}  ({100 * len(overlap) / k:.1f}%)")

spearman_norm = stats.spearmanr(comic_scores_norm.reshape(-1), coco_scores_norm.reshape(-1))
wilcoxon_norm = stats.wilcoxon(coco_scores_norm.reshape(-1), comic_scores_norm.reshape(-1))
print(f"\nSpearman rho (normalized): {spearman_norm.correlation:.3f}  (p={spearman_norm.pvalue:.3e})")
print(f"Wilcoxon (normalized, coco vs comics): statistic={wilcoxon_norm.statistic:.1f}  p={wilcoxon_norm.pvalue:.3e}")


NORMALIZED (area-fair) scores
 top-k   comics mean   coco mean      winner
     1          4.30        2.87      comics
    10          3.86        1.38      comics
    50          2.47        0.68      comics
   100          1.82        0.48      comics

Overall mean normalized score | comics: 0.286  coco: 0.083
Overall max  normalized score | comics: 4.296  coco: 2.872
(1.0 = attention proportional to region size / chance; higher = more concentrated than chance)

top-10   overlap (normalized): 3/10  (30.0%)

top-50   overlap (normalized): 25/50  (50.0%)

top-100  overlap (normalized): 50/100  (50.0%)

Spearman rho (normalized): 0.768  (p=5.204e-225)
Wilcoxon (normalized, coco vs comics): statistic=107113.0  p=2.916e-88


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharex=True, sharey=True)
vmax = np.percentile(np.concatenate([comic_scores_norm.reshape(-1), coco_scores_norm.reshape(-1)]), 99)
im0 = axes[0].imshow(comic_scores_norm, aspect="auto", cmap="magma", vmin=0, vmax=vmax)
axes[0].set_title("Comics — normalized (area-fair) vir score")
axes[0].set_xlabel("Head"); axes[0].set_ylabel("Layer")
im1 = axes[1].imshow(coco_scores_norm, aspect="auto", cmap="magma", vmin=0, vmax=vmax)
axes[1].set_title("COCO — normalized (area-fair) vir score")
axes[1].set_xlabel("Head")
fig.colorbar(im1, ax=axes, fraction=0.025, pad=0.02, label="Normalized vir score (x chance)")
save_figure_pdf(fig, outputs.figures_dir / "comics_vs_coco_vis_head_score_maps_normalized.pdf")
plt.show()

print(f"Raw-score dataset preference    : {'comics' if comic_scores.mean() > coco_scores.mean() else 'coco'}")
print(f"Normalized-score dataset preference: {'comics' if comic_scores_norm.mean() > coco_scores_norm.mean() else 'coco'}")


Raw-score dataset preference    : comics
Normalized-score dataset preference: comics


## Part 4 — Causal-effect comparison

Attention *scores* only measure correlation between a head and the target. To ask
which dataset shows a **larger causal effect**, we take a shared pool of top vir
heads (by the fair, normalized ranking — averaged across both datasets so neither
dataset's preference biases which heads get tested) and **ablate** them: force just
those heads' attention away from the target region (nothing else touched) while
generating.

A first pass at `N_CAUSAL_HEADS=50` saturated: ablation flipped ~95% of outputs on
*both* datasets, with overlapping confidence intervals — a strong intervention that
tells us "yes, causal" but can't discriminate *how much more* causal one dataset is
than the other (a ceiling effect, not a null result). We fix that two ways:

1. **Sweep the head budget** (`HEAD_BUDGETS`) down to a handful of heads, where the
   binary change-rate has room to differentiate instead of both saturating near 1.0.
2. **Use a continuous, semantic effect size**, not just the binary "did it change"
   flag or lexical overlap: mean `1 - semantic_similarity(baseline, ablated)`
   (`vis_head/judge.py`) — a fixed, VLM-independent NLI model (DeBERTa-xlarge-MNLI)
   scores how much the two texts mutually entail each other, so 0 means they say the
   same thing and 1 means they say unrelated or contradictory things, regardless of
   exact wording. This is more robust than word overlap (which conflates paraphrase
   with real change) and stays informative even when the binary rate is pinned at
   ~1.0; it also lets us run a Mann-Whitney U test (rather than comparing two
   saturated proportions) for whether comics vs. COCO differ.

In [12]:
HEAD_BUDGETS = [50, 15, 5]   # shrink the intervention until the binary rate stops saturating
N_CAUSAL_SAMPLES = 40         # per dataset
CAUSAL_MAX_NEW_TOKENS = 40

# Shared head pool: rank by the AVERAGE of both datasets' normalized scores, so the
# same heads are tested for causal effect on both datasets (an apples-to-apples probe).
combined_norm_score = (comic_scores_norm.astype(np.float64) + coco_scores_norm.astype(np.float64)) / 2.0
combined_ranked = rank_heads_by_score(combined_norm_score)
print(f"Shared head pool (by combined normalized score), sweeping budgets {HEAD_BUDGETS}")


def causal_ablation_effects_comics(model, processor, comic_dirs, n_panels: int, n_samples: int,
                                    heads_by_layer: dict) -> list[dict]:
    """For each comic, ask about a random panel; baseline vs. ablated (heads_by_layer
    suppressed on that panel's tokens) generation. Returns per-sample dicts with a
    binary `changed` flag and a continuous `effect` = 1 - semantic_similarity
    (a fixed, VLM-independent NLI-based measure, comparable across checkpoints)."""
    rng = np.random.RandomState(SEED)
    results = []
    for comic_dir in tqdm(comic_dirs[:n_samples], desc="Causal ablation [comics]", leave=False):
        strip = build_strip(comic_dir, n_panels=n_panels)
        target_panel = int(rng.randint(n_panels))
        prompt = panel_query_prompt(target_panel + 1, n_panels=n_panels)
        try:
            inputs = prepare_inputs(processor, strip.strip, prompt, DEVICE)
            img_start, img_end = find_image_token_range(inputs, processor)
            region_ids, _, _ = assign_panels_to_tokens(
                image_grid_thw=inputs["image_grid_thw"], panel_widths=strip.panel_widths, spatial_merge=spatial_merge,
            )
            panel_positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=n_panels)
            target_positions = panel_positions[target_panel]
            prompt_length = int(inputs["input_ids"].shape[1])

            baseline_sequences = run_generation(model=model, inputs=inputs, max_new_tokens=CAUSAL_MAX_NEW_TOKENS)
            baseline_text = decode_generated_text(processor, baseline_sequences, prompt_length)

            # Ablation (not steering): bias the target heads' attention AWAY from
            # the target only — the inverse of intervention_positions'
            # "boost toward target" modes. Everything else is untouched.
            hook_by_layer = {
                layer_idx: make_static_attention_mask_hook(
                    head_indices=heads, suppress_positions=target_positions, boost_positions=[],
                    n_query_heads=n_heads, device=DEVICE, decode_only=False, pad_with_suppress=False,
                )
                for layer_idx, heads in heads_by_layer.items()
            }
            ablate_inputs = prepare_inputs(processor, strip.strip, prompt, DEVICE)
            handles = register_mask_hooks(model, hook_by_layer)
            try:
                ablated_sequences = run_generation(model=model, inputs=ablate_inputs, max_new_tokens=CAUSAL_MAX_NEW_TOKENS)
            finally:
                remove_handles(handles)
            ablated_text = decode_generated_text(processor, ablated_sequences, prompt_length)

            similarity = semantic_similarity(ablated_text, baseline_text, device="cpu")
            results.append({"changed": similarity < 0.85, "effect": 1.0 - similarity})
        except Exception as exc:
            print(f"Skipping {strip.name}: {exc}")
            continue
    return results


Shared head pool (by combined normalized score), sweeping budgets [50, 15, 5]


In [13]:
def causal_ablation_effects_coco(model, processor, coco_dataset, sample_indices, n_samples: int,
                                  heads_by_layer: dict) -> list[dict]:
    """COCO counterpart: ablate heads_by_layer on the ground-truth object's tokens
    and check how much the answer to "Find the <category>." changes."""
    results = []
    for idx in tqdm(sample_indices[:n_samples], desc="Causal ablation [coco]", leave=False):
        meta, gt = coco_dataset[idx]
        try:
            image = Image.open(meta["image_path"]).convert("RGB")
            inputs = prepare_inputs(processor, image, meta["instruction"], DEVICE)
            img_start, img_end = find_image_token_range(inputs, processor)
            grid_shape = get_merged_grid_shape(inputs["image_grid_thw"], spatial_merge)
            x, y, w, h = gt["bbox"]
            _, target_positions = bbox_to_token_positions((x, y, x + w, y + h), grid_shape, image.size, img_start)
            prompt_length = int(inputs["input_ids"].shape[1])

            baseline_sequences = run_generation(model=model, inputs=inputs, max_new_tokens=CAUSAL_MAX_NEW_TOKENS)
            baseline_text = decode_generated_text(processor, baseline_sequences, prompt_length)

            hook_by_layer = {
                layer_idx: make_static_attention_mask_hook(
                    head_indices=heads, suppress_positions=target_positions, boost_positions=[],
                    n_query_heads=n_heads, device=DEVICE, decode_only=False, pad_with_suppress=False,
                )
                for layer_idx, heads in heads_by_layer.items()
            }
            ablate_inputs = prepare_inputs(processor, image, meta["instruction"], DEVICE)
            handles = register_mask_hooks(model, hook_by_layer)
            try:
                ablated_sequences = run_generation(model=model, inputs=ablate_inputs, max_new_tokens=CAUSAL_MAX_NEW_TOKENS)
            finally:
                remove_handles(handles)
            ablated_text = decode_generated_text(processor, ablated_sequences, prompt_length)

            similarity = semantic_similarity(ablated_text, baseline_text, device="cpu")
            results.append({"changed": similarity < 0.85, "effect": 1.0 - similarity})
        except Exception as exc:
            print(f"Skipping sample {idx}: {exc}")
            continue
    return results


causal_results = {}   # budget -> {"comics": [...], "coco": [...]}
for budget in HEAD_BUDGETS:
    heads_by_layer = group_heads_by_layer([(row["layer"], row["head"]) for row in combined_ranked[:budget]])
    print(f"\n=== head budget = {budget} ({sum(len(v) for v in heads_by_layer.values())} heads, "
          f"{len(heads_by_layer)} layers) ===")
    comics_res = causal_ablation_effects_comics(model, processor, comic_dirs, N_PANELS, N_CAUSAL_SAMPLES, heads_by_layer)
    coco_res = causal_ablation_effects_coco(model, processor, coco_dataset, coco_indices, N_CAUSAL_SAMPLES, heads_by_layer)
    causal_results[budget] = {"comics": comics_res, "coco": coco_res}

del model, processor
gc.collect()
torch.cuda.empty_cache()



=== head budget = 50 (50 heads, 11 layers) ===


Causal ablation [comics]:   0%|          | 0/40 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/776 [00:00<?, ?it/s]

Causal ablation [coco]:   0%|          | 0/40 [00:00<?, ?it/s]


=== head budget = 15 (15 heads, 6 layers) ===


Causal ablation [comics]:   0%|          | 0/40 [00:00<?, ?it/s]

Causal ablation [coco]:   0%|          | 0/40 [00:00<?, ?it/s]


=== head budget = 5 (5 heads, 4 layers) ===


Causal ablation [comics]:   0%|          | 0/40 [00:00<?, ?it/s]

Causal ablation [coco]:   0%|          | 0/40 [00:00<?, ?it/s]

In [14]:
print(f"{'heads':>6s}  {'dataset':>7s}  {'n':>4s}  {'change rate':>12s}  {'95% CI':>16s}  "
      f"{'mean effect':>12s}")
mw_by_budget = {}
for budget in HEAD_BUDGETS:
    for name in ("comics", "coco"):
        res = causal_results[budget][name]
        changed = [r["changed"] for r in res]
        effects = [r["effect"] for r in res]
        ci = bootstrap_ci(changed)
        print(f"{budget:6d}  {name:>7s}  {ci['n']:4d}  {ci['accuracy']:12.3f}  "
              f"[{ci['ci_low']:.3f}, {ci['ci_high']:.3f}]  {np.mean(effects):12.3f}")
    comics_effects = [r["effect"] for r in causal_results[budget]["comics"]]
    coco_effects = [r["effect"] for r in causal_results[budget]["coco"]]
    mw = stats.mannwhitneyu(comics_effects, coco_effects, alternative="two-sided")
    mw_by_budget[budget] = mw
    higher = "comics" if np.mean(comics_effects) > np.mean(coco_effects) else "coco"
    print(f"         Mann-Whitney U (continuous effect, comics vs coco): "
          f"U={mw.statistic:.1f}  p={mw.pvalue:.3e}  -> larger mean effect on {higher}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for name, color in (("comics", "tab:orange"), ("coco", "tab:green")):
    rates = [bootstrap_ci([r["changed"] for r in causal_results[b][name]])["accuracy"] for b in HEAD_BUDGETS]
    means = [np.mean([r["effect"] for r in causal_results[b][name]]) for b in HEAD_BUDGETS]
    axes[0].plot(HEAD_BUDGETS, rates, marker="o", label=name, color=color)
    axes[1].plot(HEAD_BUDGETS, means, marker="o", label=name, color=color)
axes[0].set_xlabel("Ablated heads"); axes[0].set_ylabel("Change rate (binary)")
axes[0].set_title("Binary change rate vs. head budget"); axes[0].set_ylim(0, 1.05); axes[0].legend()
axes[1].set_xlabel("Ablated heads"); axes[1].set_ylabel("Mean effect (1 - Jaccard)")
axes[1].set_title("Continuous effect size vs. head budget"); axes[1].set_ylim(0, 1.05); axes[1].legend()
save_figure_pdf(fig, outputs.figures_dir / "comics_vs_coco_causal_effect_sweep.pdf")
plt.show()


 heads  dataset     n   change rate            95% CI   mean effect
    50   comics    40         0.950  [0.875, 1.000]         0.765
    50     coco    40         0.850  [0.725, 0.950]         0.523
         Mann-Whitney U (continuous effect, comics vs coco): U=1210.0  p=8.134e-05  -> larger mean effect on comics
    15   comics    40         0.800  [0.675, 0.925]         0.543
    15     coco    40         0.750  [0.600, 0.875]         0.447
         Mann-Whitney U (continuous effect, comics vs coco): U=928.0  p=2.199e-01  -> larger mean effect on comics
     5   comics    40         0.600  [0.450, 0.750]         0.338
     5     coco    40         0.650  [0.500, 0.800]         0.359
         Mann-Whitney U (continuous effect, comics vs coco): U=727.0  p=4.854e-01  -> larger mean effect on coco


## Verdict

- **Raw vir score**: read the Part 2 top-k means / Wilcoxon result — but treat a
  comics-favoring raw gap with suspicion, since comic panels are mechanically several
  times larger than COCO objects (printed caveat above).
- **Fair (area-normalized) vir score**: Part 3 divides out region size, so its
  top-k/Wilcoxon/overlap numbers are the more trustworthy read on whether comics or
  COCO produces *stronger* vir-head targeting behavior, and whether the *same* heads
  rank highest on both (top-k overlap, Spearman rho). If comics still win here after
  normalization, the gap is a real difference in targeting behavior, not a size
  artifact — worth calling out explicitly rather than papering over.
- **Causal effect**: Part 4's per-budget change-rate and continuous effect size
  (1 - word-Jaccard) say which dataset's task actually *depends* on those heads'
  attention landing on the target, as opposed to merely correlating with it. Read the
  Mann-Whitney result at the *smallest* head budget that hasn't saturated (rate not
  pinned near 0 or 1) — that's the most discriminating point in the sweep.

Rankings (raw and normalized) for both dataset sources are saved to
`logs/vis_head_discovery_compare_datasets_comics/` and
`logs/vis_head_discovery_compare_datasets_coco/`, and are usable directly by
`interactive_steering_qwen2vl.ipynb` (`VIS_HEAD_RANKING_PATH`).